# T6 M2 — BBEH Boolean Expressions with Multi-Objective Instrumentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/OpenTrace/blob/pull/61/head/examples/notebooks/t6_m2_bbeh.ipynb)

**Milestone 2 Deliverable** — Multi-objective scoring on a real LLM task

This notebook demonstrates multi-objective optimization on the **BBEH boolean_expressions** benchmark
using the **PAL (Program-Aided Language model)** strategy from Xavier’s original experiment.

Two objectives are tracked:
- **accuracy** (binary: 1.0 = correct, 0.0 = wrong)
- **execution_time_s** (wall-clock seconds for the generated Python code)

The `LangGraphGuide.get_score_dict()` method returns both metrics per example,
enabling the M2 multi-objective infrastructure to track and visualize tradeoffs.

**Requires a real LLM API key** (OpenRouter recommended, default model: `openai/gpt-5-nano`).

---

In [ ]:
import os

# -----------------------
# Load .env file (if present) so API keys are available via os.getenv()
# -----------------------
try:
    from dotenv import load_dotenv
    # Walk up from notebook dir to find .env (works locally and in Colab)
    _env_candidates = [".env", "../.env", "../../.env", "../../../.env"]
    for _ep in _env_candidates:
        if os.path.exists(_ep):
            load_dotenv(_ep, override=False)
            print(f"Loaded .env from: {os.path.abspath(_ep)}")
            break
    else:
        print("No .env file found (will use existing env vars).")
except ImportError:
    print("python-dotenv not installed (pip install python-dotenv). Using existing env vars.")

# -----------------------
# Core defaults (edit me)
# -----------------------
BBEH_TASK_NAME = os.getenv("BBEH_TASK_NAME", "bbeh_boolean_expressions")

# Data split
N_TRAIN = int(os.getenv("N_TRAIN", "20"))
N_VAL   = int(os.getenv("N_VAL", "10"))
SEED    = int(os.getenv("SEED", "0"))

# CurriculumBuffer Mode B
VALIDATE_ON_LAST_N  = int(os.getenv("VALIDATE_ON_LAST_N", "2"))
ACCUMULATION_STEPS  = int(os.getenv("ACCUMULATION_STEPS", "2"))

# Optimization loop controls
LEARNING_RETRY = int(os.getenv("LEARNING_RETRY", "20"))
MAX_ATTEMPTS   = int(os.getenv("MAX_ATTEMPTS", "10"))

SKIP_OPTIMIZATION = os.getenv("SKIP_OPTIMIZATION", "0") == "1"

# Output
OUTPUT_FOLDER = os.getenv("OUTPUT_FOLDER", "./trace_runs")

# Optional verbosity toggles
SHOW_MERMAID_GRAPH = os.getenv("SHOW_MERMAID_GRAPH", "0") == "1"
SHOW_OPT_TRACE     = os.getenv("SHOW_OPT_TRACE", "0") == "1"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Config:")
print(f"  {BBEH_TASK_NAME=}")
print(f"  {N_TRAIN=}, {N_VAL=}, {SEED=}")
print(f"  {VALIDATE_ON_LAST_N=}, {ACCUMULATION_STEPS=}")
print(f"  {LEARNING_RETRY=}, {MAX_ATTEMPTS=}")
print(f"  {SKIP_OPTIMIZATION=}")
print(f"  {OUTPUT_FOLDER=}")

In [ ]:
import os, sys, subprocess

if IN_COLAB:
    if not os.path.exists('/content/Trace'):
        print("Setting up Trace...")
        !pip install langgraph langchain langchain_openai datasets tqdm langchain_community litellm dspy black matplotlib pandas
        !git clone https://github.com/AgentOpt/OpenTrace.git /content/Trace
        %cd /content/Trace
        !git pull origin experimental && git checkout experimental
        !sed -i 's/python_requires=">=3.13"/python_requires=">=3.12"/' setup.py
        !pip install -e .
    sys.path.append('/content/Trace')
else:
    # Local: add repo root to sys.path
    _nb_dir = os.path.dirname(os.path.abspath("__file__"))
    _repo_root = os.path.abspath(os.path.join(_nb_dir, "..", ".."))
    if _repo_root not in sys.path:
        sys.path.insert(0, _repo_root)

# Clone BBEH benchmark tasks
if not os.path.exists('bbeh'):
    !git clone https://github.com/google-deepmind/bbeh.git
else:
    print("bbeh/ already exists, skipping clone.")

# Soft-import display
try:
    from IPython.display import display
except Exception:
    def display(*args, **kwargs):
        return None

print(f"{IN_COLAB=}")

In [ ]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# -----------------------
# LLM config — auto-detect from available API keys
# -----------------------
# Priority: LLM_SERVICE env var (explicit override) > OPENAI_API_KEY > OPENROUTER_API_KEY > CUSTOMLLM_API_KEY
# When OPENAI_API_KEY is available, uses gpt-5-nano directly via OpenAI (no OpenRouter prefix).

def _get_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.getenv(name)

OPENAI_API_KEY     = _get_secret("OPENAI_API_KEY")
OPENROUTER_API_KEY = _get_secret("OPENROUTER_API_KEY")
CUSTOMLLM_API_KEY  = _get_secret("CUSTOMLLM_API_KEY")
CUSTOMLLM_URL      = os.getenv("CUSTOMLLM_URL", "http://localhost:4000/")

# Auto-detect service if not explicitly set
_explicit_service = os.getenv("LLM_SERVICE")
if _explicit_service:
    LLM_SERVICE = _explicit_service
elif OPENAI_API_KEY:
    LLM_SERVICE = "openai"
elif OPENROUTER_API_KEY:
    LLM_SERVICE = "openrouter"
elif CUSTOMLLM_API_KEY:
    LLM_SERVICE = "customllm"
else:
    raise ValueError(
        "No API key found. Set OPENAI_API_KEY, OPENROUTER_API_KEY, or CUSTOMLLM_API_KEY "
        "(via env var, .env file, or Colab secret)."
    )

# Model name: OpenRouter uses "openai/gpt-5-nano" prefix, OpenAI uses "gpt-5-nano" directly
_default_model = os.getenv("LLM_GENERAL_MODEL")
if _default_model:
    LLM_GENERAL_MODEL = _default_model
elif LLM_SERVICE == "openai":
    LLM_GENERAL_MODEL = "gpt-5-nano"
else:
    LLM_GENERAL_MODEL = "openai/gpt-5-nano"

if LLM_SERVICE == "openai":
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY missing (set env var, .env file, or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
    os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY
elif LLM_SERVICE == "openrouter":
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY missing (set env var, .env file, or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_API_KEY"]  = OPENROUTER_API_KEY
elif LLM_SERVICE == "customllm":
    if not CUSTOMLLM_API_KEY:
        raise ValueError("CUSTOMLLM_API_KEY missing (set env var, .env file, or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = CUSTOMLLM_URL
    os.environ["OPENAI_API_KEY"]  = CUSTOMLLM_API_KEY
else:
    raise ValueError(f"Unknown LLM_SERVICE: {LLM_SERVICE!r}")

llm = ChatOpenAI(model_name=LLM_GENERAL_MODEL, temperature=0)

def llm_call(prompt: str, system_instructions: str = "") -> str:
    msgs = [HumanMessage(content=prompt)]
    if system_instructions:
        msgs.insert(0, SystemMessage(content=system_instructions))
    return llm.invoke(msgs).content

print("LLM ready:", {"service": LLM_SERVICE, "model": LLM_GENERAL_MODEL})

In [ ]:
import os, json, random, inspect, time
from copy import deepcopy
from typing import Dict, Tuple, Optional

# ---- Trace imports (OpenTrace / opto) ----
try:
    from opto.trace import node, bundle
    from opto.trace.bundle import FunModule
    from opto.optimizers.optoprime_v2 import OptoPrimeV2 as OptoPrime
    from opto.trainer.guide import Guide as _TraceGuide
    from opto.trainer.algorithms.basic_algorithms import Minibatch as _TraceMinibatch
except Exception as e:
    raise ImportError(
        "Could not import OpenTrace (opto.*). "
        "Make sure OpenTrace is installed and on sys.path."
    ) from e


# -----------------------
# Small helpers
# -----------------------
def set_dict(state: dict, key, value):
    (state.data if hasattr(state, "data") else state)[key] = value

def get_no_node(x):
    return x.data if hasattr(x, "data") else x

def _snapshot_params(parameters):
    snap = {}
    for p in parameters:
        try:
            snap[p.name] = deepcopy(p.data)
        except Exception:
            snap[p.name] = p.data
    return snap

def _params_changed(before, after) -> bool:
    if before.keys() != after.keys():
        return True
    for k in before.keys():
        if str(before[k]) != str(after[k]):
            return True
    return False

def _replace_in_scope_by_identity(scope: dict, old_obj, new_obj) -> list[str]:
    replaced = []
    for k, v in list(scope.items()):
        if v is old_obj:
            scope[k] = new_obj
            replaced.append(k)
    return replaced

def bind_function(func, *, trainable=True, traceable_code=True, allow_external_dependencies=True):
    """Safely bundle() a python function into a Trace FunModule (only once)."""
    if func is None or not callable(func):
        return func
    if isinstance(func, FunModule):
        return func
    fm = bundle(trainable=trainable,
                traceable_code=traceable_code,
                allow_external_dependencies=allow_external_dependencies)(func)
    try:
        fm.__signature__ = inspect.signature(fm._fun)
    except Exception:
        pass
    return fm


# -----------------------
# Guide: graph output -> (score, feedback) + multi-objective score_dict
# -----------------------
class LangGraphGuide(_TraceGuide):
    """Guide for LangGraph-based agents with multi-objective scoring.

    get_feedback() returns the original scalar (accuracy) score and feedback string.
    get_score_dict() returns {accuracy, execution_time_s} for multi-objective tracking.

    The execution_time_s is populated per-call via _last_execution_time_s,
    set externally after each graph invocation from the instrumented execute_code() node.
    """

    def __init__(self, feedback_func, *, answer_key="final_answer", allowed_answer_set=None):
        self.feedback_func = feedback_func
        self.answer_key = answer_key
        self.allowed = allowed_answer_set
        self._last_execution_time_s = 0.0

    def _extract_answer(self, response):
        """Extract the answer value from a graph response."""
        try:
            if isinstance(response, dict) or (hasattr(response, "data") and isinstance(get_no_node(response), dict)):
                return get_no_node(get_no_node(response)[self.answer_key])
            else:
                return get_no_node(response)
        except Exception:
            return get_no_node(response)

    def get_feedback(self, query, response, reference, **kwargs):
        extracted = self._extract_answer(response)
        if self.allowed is not None:
            ok, fb = self.feedback_func(extracted, reference, self.allowed)
        else:
            ok, fb = self.feedback_func(extracted, reference)
        return float(bool(ok)), fb

    def get_score_dict(self, query, response, reference=None, **kwargs) -> Dict[str, float]:
        """Return multi-objective scores: accuracy and execution time.

        accuracy: 1.0 if correct, 0.0 if wrong (to maximize)
        execution_time_s: wall-clock seconds for code execution (to minimize)
        """
        extracted = self._extract_answer(response)
        if self.allowed is not None:
            ok, _ = self.feedback_func(extracted, reference, self.allowed)
        else:
            ok, _ = self.feedback_func(extracted, reference)
        return {
            "accuracy": float(bool(ok)),
            "execution_time_s": self._last_execution_time_s,
        }

    def copy(self):
        g = LangGraphGuide(self.feedback_func, answer_key=self.answer_key, allowed_answer_set=self.allowed)
        g._last_execution_time_s = self._last_execution_time_s
        return g


# -----------------------
# CurriculumBuffer
# -----------------------
class CurriculumBuffer:
    """Mode A (fixed pool) if training_pool is provided; Mode B (curriculum) otherwise."""
    def __init__(self, training_pool=None, *, history_size=2, sample_with_replacement=True, seed=None):
        self.pool = list(training_pool) if training_pool else []
        self.history = []
        self.history_size = int(history_size)
        self.replacement = bool(sample_with_replacement)
        self._rng = random.Random(seed)

    @property
    def is_fixed_pool(self) -> bool:
        return len(self.pool) > 0

    def add_success(self, example: dict):
        self.history.append(example)
        if len(self.history) > self.history_size:
            self.history.pop(0)

    def sample_batch(self, batch_size: int, *, current_question=None, current_solution=None) -> list[dict]:
        if self.is_fixed_pool:
            k = batch_size if self.replacement else min(batch_size, len(self.pool))
            return self._rng.choices(self.pool, k=k) if self.replacement else self._rng.sample(self.pool, k=k)
        batch = []
        max_steps = min(batch_size, 1 + len(self.history))
        for i in range(max_steps):
            if i == 0:
                batch.append({"question": current_question, "solution": current_solution})
            else:
                ex = self.history[-i]
                batch.append({"question": ex["question"], "solution": ex.get("solution", ex.get("answer"))})
        return batch


# -----------------------
# Trainer
# -----------------------
class LangGraphTrainer(_TraceMinibatch):
    def __init__(self, *, graph_root_function: str, graph_agents_functions: list[str], scope: dict,
                 optimizer, parameters: list):
        object.__init__(self)
        self.root_name = graph_root_function
        self.agent_names = list(graph_agents_functions)
        self.scope = scope
        self.optimizer = optimizer
        self.parameters = list(parameters)
        self._original_root = scope[graph_root_function]
        self._original_agents = {n: scope[n] for n in graph_agents_functions if n in scope}

    def restore_originals(self):
        self.scope[self.root_name] = self._original_root
        for name, orig in self._original_agents.items():
            self.scope[name] = orig

    def _check_corruption(self) -> bool:
        restored = False
        for name in self.agent_names:
            agent = self.scope.get(name)
            if isinstance(agent, FunModule) and getattr(agent, "_fun", None) is None:
                print(f"\u26a0\ufe0f corruption: '{name}' has ._fun=None. Restoring original.")
                self.scope[name] = self._original_agents[name]
                restored = True
        return restored

    def _run_one(self, question, solution, guide: LangGraphGuide):
        answer_key = guide.answer_key
        try:
            answer = self.scope[self.root_name](question)
            score, feedback = guide.get_feedback(question, answer, solution)
            ok = score >= 1.0
        except Exception as e:
            ok = False
            feedback = f"ERROR: {e}"
            answer = {answer_key: node("DUMMY_ANSWER")}
        return answer, ok, feedback

    def train(self, *, guide: LangGraphGuide, buffer: CurriculumBuffer,
              question=None, solution=None,
              target_updates=20, max_attempts=10, batch_size=3,
              test_optimization=True, stop_on_success=True,
              run_dir=".", save_steps=True,
              validation_set=None):
        if validation_set is None:
            validation_set = []

        answer_key = guide.answer_key
        best_state = None
        last_state = None
        history = []
        modified = False
        updates_done = 0
        global_attempt = 0

        os.makedirs(run_dir, exist_ok=True)

        while updates_done < int(target_updates):
            step_attempt = 0
            step_changed = False

            while step_attempt < int(max_attempts) and not step_changed:
                step_attempt += 1
                global_attempt += 1
                attempt = global_attempt
                print(f"[opt] attempt={attempt} update_step={updates_done+1}/{target_updates} try={step_attempt}/{max_attempts}")

                self.optimizer.zero_feedback()

                batch_examples = buffer.sample_batch(
                    int(batch_size),
                    current_question=question,
                    current_solution=solution,
                )

                answers = []
                feedbacks = []
                batch_all_correct = True

                for ex in batch_examples:
                    eq = ex["question"]
                    es = ex.get("solution", ex.get("answer"))
                    ans, ok, fb = self._run_one(eq, es, guide)
                    batch_all_correct = batch_all_correct and ok
                    answers.append(ans)
                    feedbacks.append(fb)

                if len(feedbacks) == 1:
                    common_feedback = feedbacks[0]
                else:
                    common_feedback = "\n".join([f"Feedback #{i+1}: {fb}" for i, fb in enumerate(feedbacks)])

                for ans in answers:
                    ans_node = ans.get(answer_key, ans) if isinstance(ans, dict) else ans
                    if not hasattr(ans_node, "backward"):
                        ans_node = node(str(ans_node))
                    self.optimizer.backward(
                        ans_node,
                        common_feedback,
                        visualize=bool(SHOW_OPT_TRACE),
                        print_limit=30,
                    )

                before = _snapshot_params(self.parameters)
                self.optimizer.step(verbose=True)
                after = _snapshot_params(self.parameters)
                step_changed = _params_changed(before, after)

                if self._check_corruption():
                    step_changed = False

                if not step_changed:
                    print("[opt] no parameter change, retrying...")
                    continue

                updates_done += 1
                modified = True
                last_state = {p.name: p.data for p in self.parameters}

                val_acc = None
                if validation_set:
                    n_ok = 0
                    for v in validation_set:
                        _, vok, _ = self._run_one(v["question"], v.get("solution", v.get("answer")), guide)
                        n_ok += int(vok)
                    val_acc = n_ok / float(len(validation_set))

                if save_steps:
                    try:
                        step_path = os.path.join(run_dir, f"step_{updates_done:03d}_state.txt")
                        with open(step_path, "w") as f:
                            for nm, val in last_state.items():
                                f.write(f"{nm}: {val}\n")
                    except Exception as e:
                        print(f"\u26a0\ufe0f could not save step state: {e}")

                if test_optimization and question is not None:
                    _, cur_ok, cur_fb = self._run_one(question, solution, guide)
                    val_ok = True
                    for v in validation_set:
                        _, vok, _ = self._run_one(v["question"], v.get("solution", v.get("answer")), guide)
                        if not vok:
                            val_ok = False
                            break
                    if cur_ok and val_ok:
                        best_state = last_state
                        print("[opt] gate PASS:", cur_fb)
                        if stop_on_success:
                            hist_entry = {
                                "update_step": updates_done,
                                "attempt": attempt,
                                "batch_size": int(batch_size),
                                "mode": "fixed" if buffer.is_fixed_pool else "curriculum",
                                "train_batch_all_correct": batch_all_correct,
                                "val_acc": val_acc,
                                "gate_pass": True,
                            }
                            history.append(hist_entry)
                            with open(os.path.join(run_dir, "history.jsonl"), "a") as f:
                                f.write(json.dumps(hist_entry, default=str) + "\n")
                            return modified, history, best_state, last_state

                hist_entry = {
                    "update_step": updates_done,
                    "attempt": attempt,
                    "batch_size": int(batch_size),
                    "mode": "fixed" if buffer.is_fixed_pool else "curriculum",
                    "train_batch_all_correct": batch_all_correct,
                    "val_acc": val_acc,
                    "gate_pass": bool(best_state is not None),
                }
                history.append(hist_entry)
                try:
                    with open(os.path.join(run_dir, "history.jsonl"), "a") as f:
                        f.write(json.dumps(hist_entry, default=str) + "\n")
                except Exception:
                    pass

                if stop_on_success and best_state is not None:
                    return modified, history, best_state, last_state

            if not step_changed:
                print(f"\u26a0\ufe0f stopping early: couldn't get a parameter update after {max_attempts} tries.")
                break

        return modified, history, best_state, last_state


# -----------------------
# optimize_langgraph (thin facade)
# -----------------------
def optimize_langgraph(
    *,
    graph_root_function: str,
    graph_agents_functions: list[str],
    question: str,
    solution: str,
    graph_prompts_list=None,
    answer_feedback_func=None,
    allowed_answer_set=None,
    answer_key="final_answer",
    validation_set=None,
    training_pool=None,
    batch_size=None,
    accumulation_steps=1,
    sample_with_replacement=True,
    seed=None,
    updating_steps=None,
    retry=5,
    max_attempts=10,
    stop_on_success=True,
    test_optimization=True,
    train_graph_agents_functions=True,
    memory_size=1,
    save_steps=True,
    dump_prefix="",
    output_folder=None,
    scope=None,
    optimizer_cls=None,
    trainer_cls=None,
):
    if optimizer_cls is None:
        optimizer_cls = OptoPrime
    if trainer_cls is None:
        trainer_cls = LangGraphTrainer
    if scope is None:
        scope = globals()
    if validation_set is None:
        validation_set = []
    if seed is not None:
        random.seed(seed)

    if isinstance(scope.get(graph_root_function), FunModule):
        scope[graph_root_function] = scope[graph_root_function]._fun

    parameters = []
    for name in graph_agents_functions:
        if name not in scope:
            raise KeyError(f"'{name}' not found in scope.")
        scope[name] = bind_function(scope[name], trainable=train_graph_agents_functions)
        parameters.extend(scope[name].parameters())

    if graph_prompts_list is not None:
        for i, prompt in enumerate(list(graph_prompts_list)):
            if hasattr(prompt, "data") and hasattr(prompt, "name"):
                parameters.append(prompt)
                continue
            new_prompt = node(str(prompt), trainable=True)
            _replace_in_scope_by_identity(scope, prompt, new_prompt)
            graph_prompts_list[i] = new_prompt
            parameters.append(new_prompt)

    if not parameters:
        raise ValueError("No trainable parameters found (agents/prompts list is empty).")

    opt = optimizer_cls(
        parameters,
        memory_size=memory_size,
        objective=[
            "Improve the agent so it solves the task reliably.",
            "Prefer simple, robust edits to prompts/code."
        ],
    )

    guide = LangGraphGuide(
        feedback_func=answer_feedback_func,
        answer_key=answer_key,
        allowed_answer_set=allowed_answer_set,
    )

    effective_batch_size = int(batch_size) if batch_size is not None else max(1, 1 + int(accumulation_steps))

    buffer = CurriculumBuffer(
        training_pool=training_pool,
        history_size=max(len(validation_set), 2) if validation_set else 2,
        sample_with_replacement=sample_with_replacement,
        seed=seed,
    )
    if (not buffer.is_fixed_pool) and validation_set:
        for v in validation_set:
            buffer.add_success(v)

    target_updates = int(updating_steps) if updating_steps is not None else int(retry)
    _max_attempts = int(max_attempts)

    base_dir = output_folder or "."
    os.makedirs(base_dir, exist_ok=True)
    run_name = (
        f"{dump_prefix}{graph_root_function}"
        f"__mode-{'fixed' if buffer.is_fixed_pool else 'curr'}"
        f"__bs{effective_batch_size}"
        f"__updates{target_updates}"
        f"__maxA{_max_attempts}"
        f"__mem{memory_size}"
        f"__seed{seed if seed is not None else 'none'}"
    )
    run_dir = os.path.join(base_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    trainer = trainer_cls(
        graph_root_function=graph_root_function,
        graph_agents_functions=graph_agents_functions,
        scope=scope,
        optimizer=opt,
        parameters=parameters,
    )
    modified, history, best_state, last_state = trainer.train(
        guide=guide,
        buffer=buffer,
        question=question,
        solution=solution,
        target_updates=target_updates,
        max_attempts=_max_attempts,
        batch_size=effective_batch_size,
        test_optimization=test_optimization,
        stop_on_success=stop_on_success,
        save_steps=save_steps,
        run_dir=run_dir,
        validation_set=validation_set,
    )

    chosen_state = best_state if best_state is not None else last_state
    dump_filename = None
    if modified and chosen_state is not None:
        dump_filename = os.path.join(run_dir, "best_state.txt")
        with open(dump_filename, "w") as f:
            for nm, val in chosen_state.items():
                f.write(f"{nm}: {val}\n")

    if (not test_optimization) or (best_state is None):
        trainer.restore_originals()

    return modified, dump_filename, history, chosen_state, run_dir


print("Framework loaded: LangGraphGuide (with get_score_dict), CurriculumBuffer, LangGraphTrainer, optimize_langgraph")

In [ ]:
import re, time
from langgraph.graph import StateGraph, START, END

# -----------------------
# Strategy: PAL (Program-Aided Language model)
# -----------------------
prompt_parse_problem = node(
    "Read the problem and write Python code that sets a variable named `result` to the final answer.\n"
    "- Output ONLY valid Python (no markdown fences).\n"
    "- If the task is multiple-choice, set result to the option label exactly (e.g., '(A)').\n\n"
    "Problem:\n",
    trainable=True,
    description="PAL prompt that generates python code producing a `result`."
)

# Global variable to capture execution time from the graph invocation.
# This is read by run_solver_on_example() to populate the guide's score_dict.
_last_exec_time_s = 0.0

def parse_problem(state: dict):
    question = get_no_node(state.get("question", ""))
    prompt = prompt_parse_problem + question
    code_str = llm_call(get_no_node(prompt))
    return {"code": code_str.strip(), "question": question}

def execute_code(state: dict):
    """Execute LLM-generated Python code, measuring wall-clock time.

    M2 instrumentation: wraps the exec() call with time.perf_counter()
    so execution_time_s is available as a second objective.
    """
    global _last_exec_time_s

    def strip_python_tags(code: str) -> str:
        return re.sub(
            r'(?s)(?:.*?```(?:python)?\s*\n(.*?)(?:\n```.*)?|(.*))',
            lambda m: m.group(1) if m.group(1) is not None else m.group(2),
            code,
        )

    update = {}
    try:
        code_to_run = strip_python_tags(get_no_node(state.get("code", "")))
        local_vars = {}

        # --- M2: measure wall-clock time for code execution ---
        t0 = time.perf_counter()
        exec(code_to_run, {}, local_vars)  # noqa: S102 - intentional PAL strategy
        t1 = time.perf_counter()
        _last_exec_time_s = t1 - t0

        local_vars.pop("__builtins__", None)

        if "result" in local_vars:
            update["final_answer"] = node(local_vars["result"])
        elif len(local_vars) == 1:
            update["final_answer"] = node(next(iter(local_vars.values())))
        else:
            update["final_answer"] = node(None)

    except Exception as e:
        _last_exec_time_s = 0.0
        update["final_answer"] = node(None)
        update["error"] = str(e)

    return update

def create_graph_solve_with_PAL_Strategy():
    g = StateGraph(dict)
    g.add_node("parse", parse_problem)
    g.add_node("calculate", execute_code)
    g.add_edge(START, "parse")
    g.add_edge("parse", "calculate")
    g.add_edge("calculate", END)
    return g

def solve_with_PAL_Strategy(problem: str) -> dict:
    global _last_exec_time_s
    _last_exec_time_s = 0.0  # reset before each invocation

    g = create_graph_solve_with_PAL_Strategy()
    compiled = g.compile()

    if SHOW_MERMAID_GRAPH:
        try:
            from IPython.display import Image, display
            display(Image(compiled.get_graph(xray=1).draw_mermaid_png()))
        except Exception:
            pass

    result = compiled.invoke({"question": get_no_node(problem)})
    if "final_answer" not in result:
        return {"final_answer": node("No solution found")}
    if isinstance(result["final_answer"], str):
        return {"final_answer": node(result["final_answer"])}
    return result

# Default graph spec
GRAPH_ROOT = "solve_with_PAL_Strategy"
GRAPH_AGENTS = ["parse_problem", "execute_code"]
GRAPH_PROMPTS = [prompt_parse_problem]

print("PAL strategy loaded (with execution time instrumentation).")
print("execute_code() measures wall-clock time for exec() via time.perf_counter().")

In [ ]:
import os, json, random, string

# -----------------------
# BBEH dataset loader
# -----------------------
def _find_bbeh_tasks_dir() -> str:
    candidates = [
        "bbeh/benchmark_tasks",
        "bbeh/bbeh/benchmark_tasks",
        "benchmark_tasks",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(
        "Could not locate BBEH benchmark_tasks folder.\n"
        "Clone the repo first, e.g. `git clone https://github.com/google-deepmind/bbeh.git`."
    )

bbeh_tasks_dir = _find_bbeh_tasks_dir()
print("BBEH tasks dir:", bbeh_tasks_dir)

LIMITED_BBEH_OUTPUT_TASKS = {
    "bbeh_boolean_expressions": {"(A)", "(B)", "(C)", "(D)", "(E)"},
}

def normalize_answer(ans) -> str:
    if ans is None:
        return ""
    ans = str(ans).strip().lower()
    ans = ans.translate(str.maketrans("", "", string.punctuation))
    ans = ans.replace(" ", "")
    return ans

def feedback_answer_bbeh(predicted, target, allowed_set=None):
    pred_raw = get_no_node(predicted)
    pred_norm = normalize_answer(pred_raw)
    target_norm = normalize_answer(target)

    allowed_norm = None
    if allowed_set:
        allowed_norm = {normalize_answer(a) for a in allowed_set}

    if pred_norm == target_norm:
        return True, f"SUCCESS: '{pred_raw}'"
    msg = f"FAILED: '{pred_raw}' != '{target}'. Fix the code/prompt to solve similar problems."
    if allowed_norm is not None and pred_norm not in allowed_norm:
        msg += f" (final answer must be one of: {sorted(allowed_set)})"
    return False, msg

def load_bbeh_examples(task_name: str, *, n_train: int, n_val: int, seed: int = 0):
    task_path = os.path.join(bbeh_tasks_dir, task_name, "task.json")
    if not os.path.exists(task_path):
        raise FileNotFoundError(f"Task not found: {task_path}")

    with open(task_path, "r") as f:
        task = json.load(f)

    examples = task.get("examples", [])
    rng = random.Random(seed)
    rng.shuffle(examples)

    allowed = LIMITED_BBEH_OUTPUT_TASKS.get(task_name)
    def _format_q(q: str) -> str:
        if allowed:
            return q + f"\n\nAllowed final answer: {sorted(allowed)}"
        return q

    items = [{"question": _format_q(ex["input"]), "solution": ex["target"]} for ex in examples]

    train = items[:n_train]
    val   = items[n_train:n_train + n_val]
    return train, val, allowed

train_set, val_set, allowed_set = load_bbeh_examples(
    BBEH_TASK_NAME,
    n_train=N_TRAIN,
    n_val=N_VAL,
    seed=SEED,
)

print(f"Loaded {len(train_set)} train and {len(val_set)} val examples for {BBEH_TASK_NAME}")

---
## Training Loop with Multi-Objective Metric Collection

The training loop below is identical to Xavier's original curriculum training,
with one addition: after each example (train or val), we collect `get_score_dict()`
from the `LangGraphGuide` to record both **accuracy** and **execution_time_s**.

This per-example metric log feeds the graphs in the next section.

In [ ]:
from typing import List, Dict, Tuple
import time

# -----------------------
# Multi-objective instrumented solver + evaluator
# -----------------------

# Build the guide with multi-objective support
guide = LangGraphGuide(
    feedback_func=feedback_answer_bbeh,
    answer_key="final_answer",
    allowed_answer_set=allowed_set,
)

def run_solver_on_example(ex: dict) -> Tuple[bool, str, str, Dict[str, float]]:
    """Run solver and return (ok, pred, feedback, score_dict).

    score_dict contains {accuracy, execution_time_s} from get_score_dict().
    """
    global _last_exec_time_s
    _last_exec_time_s = 0.0

    out = solve_with_PAL_Strategy(ex["question"])
    pred = get_no_node(out.get("final_answer"))
    ok, fb = feedback_answer_bbeh(pred, ex["solution"], allowed_set)

    # Populate guide's execution time from the global, then get score_dict
    guide._last_execution_time_s = _last_exec_time_s
    score_dict = guide.get_score_dict(ex["question"], out, ex["solution"])

    return ok, str(pred), fb, score_dict

def evaluate(examples: List[dict], *, name: str) -> Tuple[float, List[Dict[str, float]]]:
    """Evaluate examples, returning (accuracy, list of score_dicts)."""
    n_ok = 0
    all_score_dicts = []
    for i, ex in enumerate(examples, 1):
        ok, pred, fb, sd = run_solver_on_example(ex)
        n_ok += int(ok)
        all_score_dicts.append(sd)
        print(f"[{name}] {i:02d}/{len(examples)} ok={ok} pred={pred} "
              f"exec_time={sd['execution_time_s']:.4f}s :: {fb}")
    acc = n_ok / max(1, len(examples))
    mean_time = sum(sd['execution_time_s'] for sd in all_score_dicts) / max(1, len(all_score_dicts))
    print(f"[{name}] accuracy = {acc:.3f} ({n_ok}/{len(examples)}), mean exec_time = {mean_time:.4f}s")
    return acc, all_score_dicts


# =====================================================================
# Baseline evaluation
# =====================================================================
print("=" * 60)
print("BASELINE evaluation on validation set")
print("=" * 60)
baseline_acc, baseline_score_dicts = evaluate(val_set, name="baseline/val")

# =====================================================================
# Per-step metric collection during curriculum training
# =====================================================================
# Stores {step, phase, accuracy, execution_time_s, example_idx} per observation
metric_log = []
step_counter = 0

# Record baseline metrics
for i, sd in enumerate(baseline_score_dicts):
    metric_log.append({
        "step": 0,
        "phase": "baseline",
        "example_idx": i,
        **sd,
    })

# =====================================================================
# Curriculum training (Mode B) with metric collection
# =====================================================================
if SKIP_OPTIMIZATION:
    print("SKIP_OPTIMIZATION=1 -> skipping optimization/training.")
else:
    last_successes: List[dict] = []

    for idx, ex in enumerate(train_set, 1):
        step_counter += 1
        ok, pred, fb, sd = run_solver_on_example(ex)
        print(f"[train] {idx:02d}/{len(train_set)} ok={ok} pred={pred} "
              f"exec_time={sd['execution_time_s']:.4f}s :: {fb}")

        # Log pre-optimization metric
        metric_log.append({
            "step": step_counter,
            "phase": "train_pre",
            "example_idx": idx - 1,
            **sd,
        })

        if ok:
            last_successes.append(ex)
            last_successes = last_successes[-VALIDATE_ON_LAST_N:]
            continue

        # Optimize on the failing example
        modified, dump_file, history, chosen_state, run_dir = optimize_langgraph(
            graph_root_function=GRAPH_ROOT,
            graph_agents_functions=GRAPH_AGENTS,
            graph_prompts_list=GRAPH_PROMPTS,
            question=ex["question"],
            solution=ex["solution"],
            answer_feedback_func=feedback_answer_bbeh,
            allowed_answer_set=allowed_set,
            validation_set=last_successes,
            accumulation_steps=ACCUMULATION_STEPS,
            retry=LEARNING_RETRY,
            max_attempts=MAX_ATTEMPTS,
            test_optimization=True,
            stop_on_success=True,
            seed=SEED,
            dump_prefix=f"BBEH_{BBEH_TASK_NAME}__PAL__",
            output_folder=OUTPUT_FOLDER,
        )

        print("[train] optimize_langgraph:", {"modified": modified, "dump_file": dump_file, "run_dir": run_dir})
        if history:
            print("[train] last history entry:", history[-1])

        # Re-test after optimization
        ok2, pred2, fb2, sd2 = run_solver_on_example(ex)
        print(f"[train] after-opt ok={ok2} pred={pred2} "
              f"exec_time={sd2['execution_time_s']:.4f}s :: {fb2}")

        # Log post-optimization metric
        metric_log.append({
            "step": step_counter,
            "phase": "train_post",
            "example_idx": idx - 1,
            **sd2,
        })

        if ok2:
            last_successes.append(ex)
            last_successes = last_successes[-VALIDATE_ON_LAST_N:]

# =====================================================================
# Post-training evaluation
# =====================================================================
print("\n" + "=" * 60)
print("POST-TRAINING evaluation on validation set")
print("=" * 60)
final_acc, final_score_dicts = evaluate(val_set, name="final/val")

# Record final eval metrics
step_counter += 1
for i, sd in enumerate(final_score_dicts):
    metric_log.append({
        "step": step_counter,
        "phase": "final",
        "example_idx": i,
        **sd,
    })

print(f"\nSummary: baseline_val_acc={baseline_acc:.3f}, final_val_acc={final_acc:.3f}")
print(f"Total metric observations collected: {len(metric_log)}")

---
## Multi-Objective Graphs

Three visualizations from the collected metric log:

1. **Score progression** — accuracy and execution_time_s over training steps (dual y-axis)
2. **Scatter plot** — accuracy vs execution_time_s per example (baseline vs final)
3. **Summary table** — aggregate metrics before and after training

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# Graph 1: Score Progression (dual y-axis)
# Shows mean accuracy (left axis) and mean execution_time_s (right axis)
# aggregated per training step.
# =====================================================================

# Aggregate metrics by step
steps_seen = sorted(set(m["step"] for m in metric_log))
step_acc = []
step_time = []

for s in steps_seen:
    entries = [m for m in metric_log if m["step"] == s]
    mean_acc = np.mean([e["accuracy"] for e in entries])
    mean_time = np.mean([e["execution_time_s"] for e in entries])
    step_acc.append(mean_acc)
    step_time.append(mean_time)

fig, ax1 = plt.subplots(figsize=(10, 5))

color_acc = '#1f77b4'
color_time = '#ff7f0e'

ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Mean Accuracy', fontsize=12, color=color_acc)
line1 = ax1.plot(steps_seen, step_acc, 'o-', color=color_acc, linewidth=2,
                 markersize=7, label='accuracy')
ax1.tick_params(axis='y', labelcolor=color_acc)
ax1.set_ylim(-0.05, 1.05)

ax2 = ax1.twinx()
ax2.set_ylabel('Mean Execution Time (s)', fontsize=12, color=color_time)
line2 = ax2.plot(steps_seen, step_time, 's--', color=color_time, linewidth=2,
                 markersize=7, label='execution_time_s')
ax2.tick_params(axis='y', labelcolor=color_time)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=10)

ax1.set_title('BBEH boolean_expressions \u2014 Score Progression (Multi-Objective)', fontsize=14)
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("Graph 1: Dual-axis score progression.")
print("  Left axis (blue): mean accuracy per step (higher is better).")
print("  Right axis (orange): mean execution time per step (lower is better).")
print("  Step 0 = baseline, intermediate = training, last = final eval.")

In [ ]:
# =====================================================================
# Graph 2: Scatter — Accuracy vs Execution Time (baseline vs final)
# Each point = one validation example.
# Jitter on accuracy axis to separate overlapping correct/incorrect dots.
# =====================================================================

baseline_entries = [m for m in metric_log if m["phase"] == "baseline"]
final_entries = [m for m in metric_log if m["phase"] == "final"]

fig, ax = plt.subplots(figsize=(8, 6))

rng = np.random.RandomState(42)

if baseline_entries:
    b_acc = np.array([e["accuracy"] for e in baseline_entries])
    b_time = np.array([e["execution_time_s"] for e in baseline_entries])
    jitter_b = rng.uniform(-0.03, 0.03, size=len(b_acc))
    ax.scatter(b_time, b_acc + jitter_b, marker='o', c='#1f77b4', s=100,
               edgecolors='black', linewidths=0.8, alpha=0.7, label='Baseline', zorder=5)

if final_entries:
    f_acc = np.array([e["accuracy"] for e in final_entries])
    f_time = np.array([e["execution_time_s"] for e in final_entries])
    jitter_f = rng.uniform(-0.03, 0.03, size=len(f_acc))
    ax.scatter(f_time, f_acc + jitter_f, marker='^', c='#2ca02c', s=120,
               edgecolors='black', linewidths=0.8, alpha=0.7, label='After Training', zorder=5)

ax.set_xlabel('Execution Time (seconds)', fontsize=12)
ax.set_ylabel('Accuracy (1=correct, 0=wrong)', fontsize=12)
ax.set_title('BBEH boolean_expressions \u2014 Accuracy vs Execution Time', fontsize=14)
ax.set_ylim(-0.15, 1.15)
ax.set_yticks([0.0, 0.5, 1.0])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Graph 2: Each point is one validation example.")
print("  Ideal: top-left (high accuracy, low execution time).")
print("  Blue circles = baseline, green triangles = after curriculum training.")
print("  Small jitter on y-axis to separate overlapping points.")

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# Summary Table: Baseline vs Final metrics
# =====================================================================

baseline_entries = [m for m in metric_log if m["phase"] == "baseline"]
final_entries = [m for m in metric_log if m["phase"] == "final"]

def agg_metrics(entries):
    if not entries:
        return {"accuracy": "N/A", "exec_time_s (mean)": "N/A",
                "exec_time_s (max)": "N/A", "n_examples": 0}
    accs = [e["accuracy"] for e in entries]
    times = [e["execution_time_s"] for e in entries]
    return {
        "accuracy": f"{np.mean(accs):.3f}",
        "exec_time_s (mean)": f"{np.mean(times):.4f}",
        "exec_time_s (max)": f"{np.max(times):.4f}",
        "n_examples": len(entries),
    }

rows = [
    {"Phase": "Baseline (val)", **agg_metrics(baseline_entries)},
    {"Phase": "After Training (val)", **agg_metrics(final_entries)},
]

# Also add training metrics if available
train_pre = [m for m in metric_log if m["phase"] == "train_pre"]
train_post = [m for m in metric_log if m["phase"] == "train_post"]
if train_pre:
    rows.append({"Phase": "Train pre-opt (all)", **agg_metrics(train_pre)})
if train_post:
    rows.append({"Phase": "Train post-opt (failed)", **agg_metrics(train_post)})

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("SUMMARY: Multi-Objective Metrics")
print("=" * 70)
print(df.to_string(index=False))

print("\n" + "=" * 70)
print("M2 BBEH NOTEBOOK COMPLETE")
print("=" * 70)
print("""
Multi-Objective Instrumentation:
  - execute_code() measures wall-clock time via time.perf_counter()
  - LangGraphGuide.get_score_dict() returns {accuracy, execution_time_s}
  - Per-example metrics collected during baseline, training, and final eval

Graphs:
  Graph 1: Dual-axis score progression (accuracy + exec time over steps)
  Graph 2: Accuracy vs execution time scatter (baseline vs final)
  Summary: Aggregate metrics table

This demonstrates the M2 multi-objective infrastructure on a real LLM task.
The same get_score_dict() interface works with BasicSearch, BeamsearchAlgorithm,
and PrioritySearch (see t6_m2_trainers.ipynb for those algorithms).
""")